# Phishing Email Detection with BERT
## Notebook 04 – Evaluation and Adversarial Error Analysis

This notebook focuses on analyzing the performance and robustness of the BERT-based
phishing email classifier. It aggregates evaluation outputs, inspects false positives
and false negatives, and summarizes adversarial attack results from TextAttack.


### Notebook objectives

- Load evaluation outputs and prediction logs for baseline and augmented models.
- Compute detailed classification metrics and generate confusion matrices.
- Identify and export false positives, false negatives, and low-confidence examples.
- Analyze the impact of adversarial attacks (BAE and TextFooler) on model predictions.
- Merge TextAttack logs and export categorized failure cases (FN, FP, joint failures) to `Summary.xlsx`.


### 1. Imports and configuration

Import analysis libraries, set up paths to evaluation and attack logs, and configure
output directories for reports and summary files.


In [1]:
import pandas as pd

# Load CSVs
df_bae = pd.read_csv("../analysis/text_attack/textattack_BAE.csv")          
df_tf = pd.read_csv("../analysis/text_attack/textattack_TextFooler.csv")    

# Fixed column names
true_col = "ground_truth_output"                     
bert_col = "original_output"                         
pred_col = "perturbed_output"                 
key_col = "original_text"                       

# Attack succeeded
def is_attack_success(df):
    return df["result_type"].astype(str).str.strip().str.lower() == "successful"

# Attack defended
def is_attack_defended(df):
    return df["result_type"].astype(str).str.strip().str.lower() == "failed"

# BAE FN/FP
bae_fn = df_bae[(df_bae[true_col] == 1) & (df_bae[pred_col] == 0)]
bae_fp = df_bae[(df_bae[true_col] == 0) & (df_bae[pred_col] == 1)]

# TextFooler FN/FP
tf_fn = df_tf[(df_tf[true_col] == 1) & (df_tf[pred_col] == 0)]
tf_fp = df_tf[(df_tf[true_col] == 0) & (df_tf[pred_col] == 1)]

# BAE/TextFooler failed
bae_failed = df_bae[is_attack_success(df_bae)]
tf_failed = df_tf[is_attack_success(df_tf)]

# Failed in both 
failed_both = pd.merge(
    bae_failed,
    tf_failed,
    on=key_col,
    suffixes=("_bae", "_tf"),
    how="inner"
)


with pd.ExcelWriter("../analysis/text_attack/Summary.xlsx", engine="xlsxwriter") as writer:
    bae_fn.to_excel(writer, sheet_name="BAE_FN", index=False)
    bae_fp.to_excel(writer, sheet_name="BAE_FP", index=False)
    tf_fn.to_excel(writer, sheet_name="TextFooler_FN", index=False)
    tf_fp.to_excel(writer, sheet_name="TextFooler_FP", index=False)
    bae_failed.to_excel(writer, sheet_name="BAE_Failed", index=False)
    tf_failed.to_excel(writer, sheet_name="TF_Failed", index=False)
    failed_both.to_excel(writer, sheet_name="Failed_Both", index=False)

print("Summary.xlsx created.")


Summary.xlsx created.
